# Two-Tower LLM Embedding Model for Hotel Recommendation Data Set

Author: Jaswanthi Mandalapu

Student ID: 019148883

**Architecture:**
- User Tower  : embed top-10 highest-rated training reviews per user
- Item Tower  : embed most-recent training reviews per hotel
- Score       : cosine similarity between user and item vectors

**Model:** `all-MiniLM-L6-v2` (SentenceTransformer) - picked this because it is free, CPU-friendly, 384-dim

In [1]:
# Install if needed
# !pip install sentence-transformers pyarrow pandas numpy duckdb

In [1]:
  import os, json, time                                                                                                                                                                                                                                                       
  import numpy as np                                                                                                                                                                                                                                                          
  import pandas as pd                                                                                                                                                                                                                                                         
  import pyarrow.parquet as pq
  import duckdb
  from sentence_transformers import SentenceTransformer                                                                                                                                                                                                                       
   
  BASE_DIR    = r"/Users/jaswanthimandalapu/Documents/Sem2/CMPE 256 (Recommender Systems)/Final Project"                                                                                                                                                                      
  DATA_DIR    = os.path.join(BASE_DIR, "llm_rec_data")
  RESULTS_DIR = os.path.join(BASE_DIR, "results")                                                                                                                                                                                                                             
  EMB_DIR     = os.path.join(RESULTS_DIR, "embeddings")
  os.makedirs(EMB_DIR, exist_ok=True)                                                                                                                                                                                                                                         
                  
  TRAIN_PATH     = os.path.join(DATA_DIR, "core",     "interactions_train.parquet")                                                                                                                                                                                           
  VALID_PATH     = os.path.join(DATA_DIR, "core",     "interactions_valid.parquet")
  TEST_PATH      = os.path.join(DATA_DIR, "core",     "interactions_test.parquet")                                                                                                                                                                                            
  NEG_PATH       = os.path.join(DATA_DIR, "training", "negative_samples.parquet")
  ITEM_META_PATH = os.path.join(DATA_DIR, "item",     "item_metadata.parquet")                                                                                                                                                                                                
  ITEM_PROF_PATH = os.path.join(DATA_DIR, "item",     "item_profiles.parquet")
  CONFIG_PATH    = os.path.join(DATA_DIR, "pipeline_config.json")                                                                                                                                                                                                             
                                                                                                                                                                                                                                                                              
  ITEM_EMB_PATH  = os.path.join(EMB_DIR, "item_embeddings.npy")                                                                                                                                                                                                               
  USER_EMB_PATH  = os.path.join(EMB_DIR, "user_embeddings.npy")                                                                                                                                                                                                               
  USER_PROF_PATH = os.path.join(EMB_DIR, "user_profiles_top10rated.parquet")                                                                                                                                                                                                  
                                                                                                                                                                                                                                                                              
  # Existing profiles built by the preprocessing pipeline                                                                                                                                                                                                                     
  EXISTING_USER_PROF = os.path.join(DATA_DIR, "user", "user_profiles.parquet")                                                                                                                                                                                                
                                                                                                                                                                                                                                                                              
  with open(CONFIG_PATH) as f:
      cfg = json.load(f)                                                                                                                                                                                                                                                      
                  
  N_USERS = cfg["n_users"]                                                                                                                                                                                                                                                    
  N_ITEMS = cfg["n_items"]
  N_NEG   = cfg["neg_samples_per_user"]                                                                                                                                                                                                                                       
                                                                                                                                                                                                                                                                              
  MODEL_NAME = "all-MiniLM-L6-v2"
                                                                                                                                                                                                                                                                              
  print(f"N_USERS : {N_USERS:,}")                                                                                                                                                                                                                                             
  print(f"N_ITEMS : {N_ITEMS:,}")
  print(f"Model   : {MODEL_NAME}") 

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


N_USERS : 1,928,038
N_ITEMS : 310,334
Model   : all-MiniLM-L6-v2


## Step 1 — Build User Profiles (Top-10 Highest-Rated Reviews)

Unlike the preprocessing pipeline (which used most-recent 10), here we use the
**top-10 highest-rated** training reviews per user. These best capture what the
user loved — stronger preference signal for the user tower.

In [2]:
if os.path.exists(USER_PROF_PATH):          
  print(f"User profiles exist — skipping.")                                                                                                                                                                                                                               
                                          
elif os.path.exists(EXISTING_USER_PROF):                                                                                                                                                                                                                                    
  print("Reusing existing user_profiles.parquet from pipeline...")
  df = pd.read_parquet(EXISTING_USER_PROF)                                                                                                                                                                                                                                
  print(f"  Columns: {df.columns.tolist()}")
                                                                                                                                                                                                                                                                          
  if "user_review_history" not in df.columns:                                                                                                                                                                                                                             
      # find the object/string column that isn't user_id
      text_col = next(                                                                                                                                                                                                                                                    
          (c for c in df.columns if df[c].dtype == object and c != "user_id"),                                                                                                                                                                                            
          None
      )                                                                                                                                                                                                                                                                   
      if text_col is None:
          raise ValueError(f"No text column found. dtypes:\n{df.dtypes}")                                                                                                                                                                                                 
      print(f"  Renaming '{text_col}' → 'user_review_history'")
      df = df.rename(columns={text_col: "user_review_history"})
                                      
  df[["user_idx", "user_id", "user_review_history"]].to_parquet(USER_PROF_PATH, index=False)                                                                                                                                                                              
  print(f"  Saved {len(df):,} users → {USER_PROF_PATH}")                                                                                                                                                                                                                  
                                                                                                                                                                                                                                                                          
else:                                                                                                                                                                                                                                                                       
  # Build from scratch: two-pass DuckDB
  print("Building user profiles from scratch (two-pass)...")                                                                                                                                                                                                              
  t0 = time.time()                    

  train_p = TRAIN_PATH.replace("\\", "/")                                                                                                                                                                                                                                 
  meta_p  = ITEM_META_PATH.replace("\\", "/")
  out_p   = USER_PROF_PATH.replace("\\", "/")                                                                                                                                                                                                                             
  tmp_dir = os.path.join(DATA_DIR, "tmp").replace("\\", "/")                                                                                                                                                                                                              
  keys_p  = tmp_dir + "/top10_keys.parquet"
  os.makedirs(tmp_dir, exist_ok=True)                                                                                                                                                                                                                                     
                                      
  con = duckdb.connect()                                                                                                                                                                                                                                                  
  con.execute("SET temp_directory='" + tmp_dir + "'")                                                                                                                                                                                                                     
  con.execute("SET memory_limit='4GB'")                                                                                                                                                                                                                                   
  con.execute("SET threads=4")                                                                                                                                                                                                                                            
  con.execute("SET preserve_insertion_order=false")
                                                                                                                                                                                                                                                                          
  if not os.path.exists(keys_p):                                                                                                                                                                                                                                          
      print("  Pass 1: ranking (numeric only)...")                                                                                                                                                                                                                        
      con.execute(                                                                                                                                                                                                                                                        
          "COPY ("                    
          "  SELECT interaction_id, user_idx FROM ("                                                                                                                                                                                                                      
          "    SELECT interaction_id, user_idx,"                                                                                                                                                                                                                          
          "           ROW_NUMBER() OVER ("
          "               PARTITION BY user_idx"                                                                                                                                                                                                                          
          "               ORDER BY rating DESC, timestamp DESC"                                                                                                                                                                                                           
          "           ) AS rn"
          "    FROM (SELECT interaction_id, user_idx, rating, timestamp"                                                                                                                                                                                                  
          "          FROM read_parquet('" + train_p + "'))"
          "  ) WHERE rn <= 10"                                                                                                                                                                                                                                            
          ") TO '" + keys_p + "' (FORMAT PARQUET)"
      )                                                                                                                                                                                                                                                                   
      print(f"  Pass 1 done: {pq.read_metadata(keys_p).num_rows:,} keys")                                                                                                                                                                                                 
  else:                                                                                                                                                                                                                                                                   
      print("  Pass 1 keys already exist — skipping.")                                                                                                                                                                                                                    
                                          
  print("  Pass 2: joining text in batches...")                                                                                                                                                                                                                           
  BATCH  = 200_000
  writer = None                                                                                                                                                                                                                                                           
  n_done = 0                          
                                                                                                                                                                                                                                                                          
  for start in range(0, N_USERS, BATCH):                                                                                                                                                                                                                                  
      end = min(start + BATCH - 1, N_USERS - 1)
                                                                                                                                                                                                                                                                          
      batch = con.execute(            
          "SELECT f.user_idx, f.user_id, COUNT(*)::INTEGER AS n_profile_reviews,"                                                                                                                                                                                         
          "  STRING_AGG("                                                                                                                                                                                                                                                 
          "    '[' || COALESCE(m.hotel_name, 'Unknown Hotel')"
          "    || ' - ' || CAST(CAST(f.rating AS INTEGER) AS VARCHAR) || '/5]'"                                                                                                                                                                                           
          "    || CASE WHEN f.title IS NOT NULL THEN chr(10) || f.title ELSE '' END"                                                                                                                                                                                      
          "    || chr(10) || f.review_text,"
          "    chr(10) || chr(10) || '---' || chr(10) || chr(10)"                                                                                                                                                                                                         
          "    ORDER BY f.rating DESC, f.timestamp DESC"                                                                                                                                                                                                                  
          "  ) AS user_review_history"                                                                                                                                                                                                                                    
          " FROM read_parquet('" + keys_p + "') k"                                                                                                                                                                                                                        
          " JOIN read_parquet('" + train_p + "') f ON k.interaction_id = f.interaction_id"                                                                                                                                                                                
          " LEFT JOIN read_parquet('" + meta_p + "') m ON f.item_idx = m.item_idx"                                                                                                                                                                                        
          " WHERE k.user_idx BETWEEN " + str(start) + " AND " + str(end) +                                                                                                                                                                                                
          "   AND f.review_text IS NOT NULL"                                                                                                                                                                                                                              
          "   AND LENGTH(TRIM(f.review_text)) > 20"                                                                                                                                                                                                                       
          " GROUP BY f.user_idx, f.user_id ORDER BY f.user_idx"                                                                                                                                                                                                           
      ).fetch_arrow_table()                                                                                                                                                                                                                                               
                                                                                                                                                                                                                                                                          
      if batch.num_rows > 0:                                                                                                                                                                                                                                              
          if writer is None:                                                                                                                                                                                                                                              
              writer = pq.ParquetWriter(out_p, batch.schema)                                                                                                                                                                                                              
          writer.write_table(batch)                                                                                                                                                                                                                                       
          n_done += batch.num_rows
                                                                                                                                                                                                                                                                          
      print(f"  {min(end+1,N_USERS):>10,} / {N_USERS:,}  |  {(time.time()-t0)/60:.1f}m", end="\r")
                                      
  if writer:                                                                                                                                                                                                                                                              
      writer.close()                                                                                                                                                                                                                                                      
  con.close()                                                                                                                                                                                                                                                             
  print(f"\nDone in {(time.time()-t0)/60:.1f} min — {n_done:,} users")  

User profiles exist — skipping.


## Step 2 — Load Profiles

In [26]:
pd.read_parquet(EXISTING_USER_PROF).head(1).to_dict()                                                                                          

{'user_idx': {0: 0},
 'user_id': {0: '----Meat-a-balls----'},
 'n_profile_reviews': {0: 10},
 'user_review_history': {0: '[Embassy Suites by Hilton Atlanta Galleria — 3/5]\nHas room for improvement\nWe stayed here twice in Sept 2013. Both times ran into a few problems. On the first visit we had to ask for a different room due to a very strong smell of cigarettes, non-working outlets, air-conditioning issues and a deadbolt installed incorrectly on the bathroom door (you could lock somebody in the bathroom instead of the other way around). The second room was better. The breakfast to order wasn\'t that great, one morning they ran out of grits 2 hours into the service so I woke up earlier the next morning in order to get the grits but they were cold and hard. The pool area was nice. The wireless system and cell phone service was weak in the rooms but strong in the lobby. I would pick a different hotel next time if returning to the area.\n\n---\n\n[Homewood Suites by Hilton Columbus — 5/5]

In [3]:
print("Loading item profiles...")
item_prof = pd.read_parquet(
    ITEM_PROF_PATH,
    columns=["item_idx", "aggregated_review_text"]
).sort_values("item_idx").reset_index(drop=True)

print("Loading user profiles (top-10 highest-rated)...")
user_prof = pd.read_parquet(
    USER_PROF_PATH,
    columns=["user_idx", "user_review_history"]
).sort_values("user_idx").reset_index(drop=True)

print(f"Item profiles : {len(item_prof):,} hotels")
print(f"User profiles : {len(user_prof):,} users")
print(f"Items missing profiles: {N_ITEMS - len(item_prof):,}  (will get zero vector)")
print(f"Users missing profiles: {N_USERS - len(user_prof):,}  (will get zero vector)")
print()
print("Sample user profile:")
print(user_prof["user_review_history"].iloc[0][:500])
print("\n---\n")
print("Sample item profile:")
print(item_prof["aggregated_review_text"].iloc[0][:500])

Loading item profiles...
Loading user profiles (top-10 highest-rated)...
Item profiles : 309,883 hotels
User profiles : 1,928,038 users
Items missing profiles: 451  (will get zero vector)
Users missing profiles: 0  (will get zero vector)

Sample user profile:
[Embassy Suites by Hilton Atlanta Galleria — 3/5]
Has room for improvement
We stayed here twice in Sept 2013. Both times ran into a few problems. On the first visit we had to ask for a different room due to a very strong smell of cigarettes, non-working outlets, air-conditioning issues and a deadbolt installed incorrectly on the bathroom door (you could lock somebody in the bathroom instead of the other way around). The second room was better. The breakfast to order wasn't that great, one mornin

---

Sample item profile:
[Rating: 5/5] Paradise
The staff is friendly and very service-oriented. Each time we hit the pool or the beach, they’d welcome us, asked about our day, asked if we needed towels, water, drinks, food, reservations

## Step 3 — Load Embedding Model

In [4]:
print(f"Loading {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME)
EMB_DIM = model.get_sentence_embedding_dimension()
print(f"Embedding dimension : {EMB_DIM}")
print(f"Max sequence length : {model.max_seq_length} tokens")
print("Note: texts longer than max_seq_length are truncated.")
print("      This affects mean item profile length (~6,000 chars) — only first ~2,000 chars used.")

Loading all-MiniLM-L6-v2 ...


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 9681.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension : 384
Max sequence length : 256 tokens
Note: texts longer than max_seq_length are truncated.
      This affects mean item profile length (~6,000 chars) — only first ~2,000 chars used.


/var/folders/z9/qf_jqt0s51n7dd3yz7l8gh840000gn/T/ipykernel_34795/528283613.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMB_DIM = model.get_sentence_embedding_dimension()


## Step 4 — Embed Items

Produces a float32 array of shape `(N_ITEMS, EMB_DIM)`.
Items without profiles get a zero vector.
Embeddings are L2-normalised so dot product = cosine similarity.

In [5]:
if os.path.exists(ITEM_EMB_PATH):
    print(f"Item embeddings already exist — loading from disk.")
    item_embeddings = np.load(ITEM_EMB_PATH)
    print(f"Loaded: {item_embeddings.shape}")
else:
    print(f"Embedding {len(item_prof):,} hotel profiles...")
    BATCH_SIZE = 64
    t0 = time.time()

    texts = item_prof["aggregated_review_text"].fillna("").tolist()

    item_vecs = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,   # L2 normalise → dot product = cosine sim
        convert_to_numpy=True,
    ).astype(np.float32)

    # Place into full array indexed by item_idx
    # Items without profiles stay as zero vector
    max_item_idx = int(item_prof["item_idx"].max()) + 1
    item_embeddings = np.zeros((max_item_idx, EMB_DIM), dtype=np.float32)
    item_embeddings[item_prof["item_idx"].values] = item_vecs

    np.save(ITEM_EMB_PATH, item_embeddings)
    print(f"\nDone in {(time.time()-t0)/60:.1f} min")
    print(f"Saved: {item_embeddings.shape}  →  {ITEM_EMB_PATH}")
    print(f"File size: {os.path.getsize(ITEM_EMB_PATH)/1e6:.0f} MB")

Item embeddings already exist — loading from disk.
Loaded: (310334, 384)


## Step 5 — Embed Users

Produces a float32 array of shape `(N_USERS, EMB_DIM)`.
Processed in chunks of 100K and saved incrementally so
you can resume if interrupted.

In [6]:
# ── Step 5 — Embed Users ──────────────────────────────────────────
PARTIAL_USER = USER_EMB_PATH + ".partial.npy"                                                                                                                                                                                                                               
CHUNK_EMB    = 100_000                      
BATCH_SIZE   = 64                                                                                                                                                                                                                                                           
                                                                                                                                                                                                                                                                          
if os.path.exists(USER_EMB_PATH):                                                                                                                                                                                                                                           
  print("User embeddings exist — loading.")                                                                                                                                                                                                                               
  user_embeddings = np.load(USER_EMB_PATH)                                                                                                                                                                                                                                
  print(f"Loaded: {user_embeddings.shape}")
else:
  # Resume from checkpoint if available                                                                                                                                                                                                                                   
  if os.path.exists(PARTIAL_USER):        
      user_embeddings = np.load(PARTIAL_USER)                                                                                                                                                                                                                             
      done_flags = (user_embeddings != 0).any(axis=1)
      last_done  = int(done_flags.nonzero()[0].max()) + 1 if done_flags.any() else 0                                                                                                                                                                                      
      print(f"Resuming from checkpoint: ~{last_done:,} users done")
  else:                                                                                                                                                                                                                                                                   
      user_embeddings = np.zeros((N_USERS, EMB_DIM), dtype=np.float32)                                                                                                                                                                                                    
      last_done = 0                                                                                                                                                                                                                                                       
                                                                                                                                                                                                                                                                          
  print(f"Embedding user profiles (streaming from parquet)...")                                                                                                                                                                                                           
  t0     = time.time()                                                                                                                                                                                                                                                    
  n_done = last_done
                                                                                                                                                                                                                                                                          
  # Stream parquet in chunks — never loads all 1.9M texts at once
  pf = pq.ParquetFile(USER_PROF_PATH)                                                                                                                                                                                                                                     
  for batch in pf.iter_batches(batch_size=CHUNK_EMB,
                                columns=["user_idx", "user_review_history"]):                                                                                                                                                                                             
      chunk_df   = batch.to_pandas()      
      chunk_idxs = chunk_df["user_idx"].values                                                                                                                                                                                                                            
              
      if chunk_idxs.max() < last_done:                                                                                                                                                                                                                                    
          continue   # skip already-done chunks on resume
                                                                                                                                                                                                                                                                          
      chunk_texts = chunk_df["user_review_history"].fillna("").tolist()                                                                                                                                                                                                   

      vecs = model.encode(                                                                                                                                                                                                                                                
          chunk_texts,
          batch_size=BATCH_SIZE,                                                                                                                                                                                                                                          
          show_progress_bar=False,        
          normalize_embeddings=True,  
          convert_to_numpy=True,
      ).astype(np.float32)                                                                                                                                                                                                                                                

      user_embeddings[chunk_idxs] = vecs                                                                                                                                                                                                                                  
      n_done += len(chunk_idxs)       

      np.save(PARTIAL_USER, user_embeddings)   # checkpoint to disk                                                                                                                                                                                                       

      elapsed = time.time() - t0                                                                                                                                                                                                                                          
      eta     = elapsed / max(n_done - last_done, 1) * (N_USERS - n_done)
      print(f"  {n_done:>10,} / {N_USERS:,}  |  {elapsed/60:.1f}m  |  ETA {eta/60:.1f}m", end="\r")
                                                                                                                                                                                                                                                                          
  np.save(USER_EMB_PATH, user_embeddings)
  if os.path.exists(PARTIAL_USER):                                                                                                                                                                                                                                        
      os.remove(PARTIAL_USER)                                                                                                                                                                                                                                             
  print(f"\nDone in {(time.time()-t0)/60:.1f} min")
  print(f"Saved: {user_embeddings.shape}  ({os.path.getsize(USER_EMB_PATH)/1e6:.0f} MB)")   

User embeddings exist — loading.
Loaded: (1928038, 384)


## Step 6 — Prepare Evaluation Data

In [7]:
print("Loading evaluation data...")

valid_df = (
    pd.read_parquet(VALID_PATH, columns=["user_idx", "item_idx"])
    .sort_values("user_idx").reset_index(drop=True)
)
test_df = (
    pd.read_parquet(TEST_PATH, columns=["user_idx", "item_idx"])
    .sort_values("user_idx").reset_index(drop=True)
)
neg_df = pd.read_parquet(NEG_PATH)   # already sorted by user_idx

# Reshape negatives to (N_USERS, 100)
neg_matrix = neg_df["neg_item_idx"].values.reshape(N_USERS, N_NEG).astype(np.int32)
valid_pos  = valid_df["item_idx"].values.astype(np.int32)
test_pos   = test_df["item_idx"].values.astype(np.int32)

# Alignment check
neg_user_order = neg_df["user_idx"].values.reshape(N_USERS, N_NEG)[:, 0]
assert np.array_equal(neg_user_order, valid_df["user_idx"].values), "Alignment mismatch!"

print(f"neg_matrix : {neg_matrix.shape}")
print(f"valid_pos  : {valid_pos.shape}")
print(f"test_pos   : {test_pos.shape}")
print("Alignment check passed.")

Loading evaluation data...
neg_matrix : (1928038, 100)
valid_pos  : (1928038,)
test_pos   : (1928038,)
Alignment check passed.


## Step 7 — Evaluation Function

In [9]:
def evaluate_two_tower(
    user_embeddings,
    item_embeddings,
    pos_items,
    neg_matrix,
    K_list=(1, 5, 10, 20),
    batch_size=10000,          # reduce this
    neg_chunk_size=20,         # NEW: chunk negatives
):
    n_users, n_neg = neg_matrix.shape
    all_ranks = np.empty(n_users, dtype=np.int32)

    t0 = time.time()
    for start in range(0, n_users, batch_size):
        end = min(start + batch_size, n_users)

        u_vecs = user_embeddings[start:end]              # (B, D)
        p_vecs = item_embeddings[pos_items[start:end]]   # (B, D)

        pos_scores = (u_vecs * p_vecs).sum(axis=1)       # (B,)
        rank_counts = np.zeros(end - start, dtype=np.int32)

        # process negatives in chunks
        for neg_start in range(0, n_neg, neg_chunk_size):
            neg_end = min(neg_start + neg_chunk_size, n_neg)

            neg_ids = neg_matrix[start:end, neg_start:neg_end]   # (B, chunk)
            neg_vecs = item_embeddings[neg_ids]                 # (B, chunk, D)

            neg_scores = np.einsum('bd,bnd->bn', u_vecs, neg_vecs)

            rank_counts += (neg_scores > pos_scores[:, None]).sum(axis=1)

        all_ranks[start:end] = 1 + rank_counts

        print(f"  {end:>10,} / {n_users:,} users scored  |  {(time.time()-t0):.0f}s", end="\r")

    print()

    metrics = {}
    for K in K_list:
        metrics[f"HR@{K}"] = round(float((all_ranks <= K).mean()), 4)
        metrics[f"NDCG@{K}"] = round(float(
            np.where(all_ranks <= K, 1.0 / np.log2(all_ranks + 1), 0.0).mean()
        ), 4)
    metrics["MRR"] = round(float((1.0 / all_ranks).mean()), 4)

    return metrics, all_ranks

## Step 8 — Evaluate on Validation Set

In [10]:
print("Evaluating Two-Tower model on VALIDATION set...")
t0 = time.time()

val_metrics, val_ranks = evaluate_two_tower(
    user_embeddings, item_embeddings,
    valid_pos, neg_matrix
)

print(f"Done in {time.time()-t0:.1f}s")
print()

col_order = ["HR@1", "HR@5", "HR@10", "HR@20", "NDCG@10", "NDCG@20", "MRR"]

# Load baseline results for comparison
baseline_path = os.path.join(RESULTS_DIR, "baselines_valid.csv")
if os.path.exists(baseline_path):
    comparison = pd.read_csv(baseline_path, index_col=0)
    comparison.loc[f"Two-Tower ({MODEL_NAME})"] = val_metrics
else:
    comparison = pd.DataFrame([val_metrics], index=[f"Two-Tower ({MODEL_NAME})"])

comparison = comparison[col_order]

print("=" * 75)
print("VALIDATION RESULTS  (101 candidates: 1 positive + 100 negatives)")
print("=" * 75)
display(comparison)

# Rank distribution
print(f"\nTwo-Tower rank stats:")
print(f"  Median rank : {np.median(val_ranks):.0f}")
print(f"  Mean rank   : {np.mean(val_ranks):.2f}")
print(f"  % rank == 1 : {(val_ranks==1).mean()*100:.2f}%")
print(f"  % rank <= 5 : {(val_ranks<=5).mean()*100:.2f}%")
print(f"  % rank <=10 : {(val_ranks<=10).mean()*100:.2f}%")

Evaluating Two-Tower model on VALIDATION set...
   1,928,038 / 1,928,038 users scored  |  77s
Done in 76.9s

VALIDATION RESULTS  (101 candidates: 1 positive + 100 negatives)


,HR@1,HR@5,HR@10,HR@20,NDCG@10,NDCG@20,MRR
Two-Tower (all-MiniLM-L6-v2),0.0656,0.1847,0.282,0.4273,0.1568,0.1933,0.142



Two-Tower rank stats:
  Median rank : 27
  Mean rank   : 33.94
  % rank == 1 : 6.56%
  % rank <= 5 : 18.47%
  % rank <=10 : 28.20%


## Step 9 — Evaluate on Test Set

In [11]:
print("Evaluating Two-Tower model on TEST set...")
t0 = time.time()

test_metrics, test_ranks = evaluate_two_tower(
    user_embeddings, item_embeddings,
    test_pos, neg_matrix
)

print(f"Done in {time.time()-t0:.1f}s")

# Load baseline test results
test_baseline_path = os.path.join(RESULTS_DIR, "baselines_test.csv")
if os.path.exists(test_baseline_path):
    test_comparison = pd.read_csv(test_baseline_path, index_col=0)
    test_comparison.loc[f"Two-Tower ({MODEL_NAME})"] = test_metrics
else:
    test_comparison = pd.DataFrame([test_metrics], index=[f"Two-Tower ({MODEL_NAME})"])

test_comparison = test_comparison[col_order]

print("=" * 75)
print("TEST RESULTS  (101 candidates: 1 positive + 100 negatives)")
print("=" * 75)
display(test_comparison)

Evaluating Two-Tower model on TEST set...
   1,928,038 / 1,928,038 users scored  |  76s
Done in 76.2s
TEST RESULTS  (101 candidates: 1 positive + 100 negatives)


,HR@1,HR@5,HR@10,HR@20,NDCG@10,NDCG@20,MRR
Two-Tower (all-MiniLM-L6-v2),0.0553,0.1645,0.2577,0.4009,0.1401,0.176,0.1278


## Step 10 — Save Results

In [12]:
# Save updated comparison tables
comparison.to_csv(os.path.join(RESULTS_DIR, "baselines_valid.csv"))
test_comparison.to_csv(os.path.join(RESULTS_DIR, "baselines_test.csv"))

print("Results saved.")
print()
print("=" * 75)
print("FINAL SUMMARY")
print("=" * 75)
summary = pd.DataFrame({
    "Validation" : val_metrics,
    "Test"       : test_metrics,
}).T[col_order]
print(f"\nTwo-Tower ({MODEL_NAME}):")
display(summary)

Results saved.

FINAL SUMMARY

Two-Tower (all-MiniLM-L6-v2):


,HR@1,HR@5,HR@10,HR@20,NDCG@10,NDCG@20,MRR
Validation,0.0656,0.1847,0.2820,0.4273,0.1568,0.1933,0.1420
Test,0.0553,0.1645,0.2577,0.4009,0.1401,0.1760,0.1278
